# φ^∞ Golden Ratio Key Exchange (GRKX) Demonstration

This notebook demonstrates the **GRKX** quantum-safe key exchange protocol built on the TUPT modular space.

**Key Properties:**
- Shared secret agreement via φ-modulated scalar multiplication
- BB84-compatible QKD basis sifting simulation
- TTT-stable nonce generation (digital root ∉ {3, 6, 9})
- 256-bit cryptographic shared secrets via SHA-256 expansion

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from phi_infinity_lattice_compression.quantum_encryption import (
    TUPT_MODULO,
    GRKXKeyPair,
    GRKXProtocol,
    _digital_root,
)

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=['#00f0ff', '#ffd700', '#ff3366', '#77dd77'])

## 1. Single Key Exchange

Alice and Bob each generate a `GRKXKeyPair`. The shared secret must be identical on both sides.

In [ ]:
alice = GRKXKeyPair()
bob = GRKXKeyPair()

print(f"Alice private key:  {alice.private_key}")
print(f"Alice public locus: {alice.public_key}")
print(f"Bob private key:    {bob.private_key}")
print(f"Bob public locus:   {bob.public_key}")

alice_secret = alice.derive_shared_secret(bob.public_key)
bob_secret = bob.derive_shared_secret(alice.public_key)

print(f"\nAlice shared secret: {alice_secret.hex()[:48]}...")
print(f"Bob shared secret:   {bob_secret.hex()[:48]}...")
print(f"\n✓ Secrets match: {alice_secret == bob_secret}")

## 2. TTT Stability Verification

We generate 1000 GRKX key pairs and verify that **every** private key has a TTT-stable digital root (∉ {3, 6, 9}).

In [ ]:
n_trials = 1000
roots = []
for _ in range(n_trials):
    kp = GRKXKeyPair()
    roots.append(_digital_root(kp.private_key))

roots_arr = np.array(roots)
chaotic_count = np.sum(np.isin(roots_arr, [3, 6, 9]))

fig, ax = plt.subplots()
counts = [np.sum(roots_arr == r) for r in range(1, 10)]
colors = ['#00f0ff' if r not in (3, 6, 9) else '#ff3366' for r in range(1, 10)]
ax.bar(range(1, 10), counts, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Digital Root', fontsize=14)
ax.set_ylabel('Count', fontsize=14)
ax.set_title(f'GRKX Private Key Digital Roots (n={n_trials}, chaotic={chaotic_count})', fontsize=15)
ax.set_xticks(range(1, 10))
for i, c in enumerate(counts):
    if c > 0:
        ax.text(i + 1, c + 5, str(c), ha='center', fontsize=10, color='white')
plt.tight_layout()
plt.show()

print(f"Chaotic nonces generated: {chaotic_count} / {n_trials} (must be 0)")

## 3. BB84 QKD Basis Sifting Simulation

We simulate the classical post-processing step of BB84 quantum key distribution using the GRKX shared secret as the raw key material.

In [ ]:
alice_s, bob_s = GRKXProtocol.execute()
assert alice_s == bob_s, "Key exchange failed!"

# Simulate basis choices for 256 bit positions
np.random.seed(42)
alice_basis = np.random.randint(0, 2, size=256).tolist()
bob_basis = np.random.randint(0, 2, size=256).tolist()

# Sift using Alice's basis
sifted_alice = GRKXProtocol.qkd_sift_basis(alice_s, alice_basis)
sifted_bob = GRKXProtocol.qkd_sift_basis(bob_s, bob_basis)

print(f"Raw shared secret length: {len(alice_s) * 8} bits")
print(f"Alice sifted key length:  {len(sifted_alice)} bits")
print(f"Bob sifted key length:    {len(sifted_bob)} bits")
print(f"\nSifted bits (Alice, first 32): {sifted_alice[:32]}")
print(f"Sifted bits (Bob, first 32):   {sifted_bob[:32]}")

## 4. Public Locus Distribution in TUPT Space

We visualize how GRKX public keys distribute across the modular ring $\mathbb{Z}_{12289}$.

In [ ]:
n_keys = 2000
public_loci = [GRKXKeyPair().public_key for _ in range(n_keys)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(public_loci, bins=50, color='#00f0ff', edgecolor='#111', alpha=0.85)
axes[0].set_xlabel('Public Locus Value', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title(f'GRKX Public Locus Distribution (n={n_keys})', fontsize=13)
axes[0].axhline(y=n_keys/50, color='#ffd700', linestyle='--', alpha=0.7, label='Uniform expected')
axes[0].legend()

# Polar spiral plot
phi = (1 + np.sqrt(5)) / 2
angles = [2 * np.pi * (loc / TUPT_MODULO) * phi for loc in public_loci[:500]]
radii = [loc / TUPT_MODULO for loc in public_loci[:500]]
ax2 = fig.add_subplot(122, projection='polar')
ax2.scatter(angles, radii, c='#ffd700', s=2, alpha=0.5)
ax2.set_title('φ-Spiral Projection', fontsize=13, pad=20)
ax2.set_facecolor('#111')

plt.tight_layout()
plt.show()

## Summary

| Property | Result |
|----------|--------|
| Shared secret agreement | ✓ Deterministic |
| TTT stability | ✓ Zero chaotic nonces |
| QKD compatibility | ✓ BB84 basis sifting |
| Key space | $\mathbb{Z}_{12289}$ (NTT-friendly prime) |
| Security | TUPT Period Immunity (Theorem 5.2) |

*Nexus Resonance Codex — GRKX Quantum Key Exchange Protocol*